In [ ]:
# ============================================================
# NAMA    : Arfal Malik Gibran
# NPM     : 24782005
# TANGGAL : 19 Mei 2026
# ============================================================
NAMA = 'Arfal Malik Gibran'
NPM  = '24782005'
my_identity = f'{NAMA} – {NPM}'
print(f'Identitas: {my_identity}')

In [ ]:
# ============================================================
# DESKRIPSI 2 — Install library yang dibutuhkan
# ============================================================
!pip install -q transformers[torch] accelerate sentencepiece

In [ ]:
# ============================================================
# DESKRIPSI 2 — Import library dan inisialisasi 3 model
# ============================================================
from transformers import pipeline
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('--- Menyiapkan Lab NLP Modern (Varian Kompatibel) ---')

# Model 1: Fill-Mask (IndoBERT)
mask_filler = pipeline('fill-mask',
                        model='cahya/bert-base-indonesian-1.5G')

# Model 2: Sentiment Analysis
sentiment_analyzer = pipeline('sentiment-analysis',
                               model='cardiffnlp/twitter-xlm-roberta-base-sentiment')

# Model 3: NER
ner_recognizer = pipeline('ner',
                           model='cahya/bert-base-indonesian-NER',
                           aggregation_strategy='simple')

print('\u2705 Semua model berhasil diinisialisasi!')

In [ ]:
# ============================================================
# Upload dataset langsung di Google Colab
# Klik tombol "Choose Files" yang muncul lalu pilih 3 file CSV
# ============================================================
from google.colab import files
import pandas as pd

uploaded = files.upload()  # pilih 3 file CSV sekaligus

df_sentiment = pd.read_csv('kalimat_sentiment_24782005.csv')
df_rumpang   = pd.read_csv('kalimat_rumpang_24782005.csv')
df_informal  = pd.read_csv('kalimat_informal_24782005.csv')

print(f'Dataset sentiment  : {len(df_sentiment)} kalimat')
print(f'  Distribusi label  : {dict(df_sentiment["label"].value_counts())}')
print(f'Dataset rumpang    : {len(df_rumpang)} kalimat')
print(f'Dataset informal   : {len(df_informal)} kalimat')
print()
print('\u2705 Semua dataset berhasil dimuat!')

In [ ]:
# ============================================================
# DESKRIPSI 3 — POIN a & b:
# Masukkan 5 kalimat rumpang ke model fill-mask
# OUTPUT TABEL: PREDIKSI | SKOR PROBABILITAS (Top 5)
# ============================================================

print('=' * 65)
print('EKSPERIMEN FILL-MASKING — 5 Kalimat Rumpang')
print('=' * 65)

for idx, row in df_rumpang.iterrows():
    kalimat = row['kalimat']
    print(f'\n[Kalimat {idx+1}]')
    print(f'Input: {kalimat}\n')

    hasil = mask_filler(kalimat)

    tabel = pd.DataFrame([
        {
            'Peringkat'        : i + 1,
            'Prediksi'         : res['token_str'],
            'Skor Probabilitas': round(res['score'], 4)
        }
        for i, res in enumerate(hasil[:5])
    ])
    display(tabel)
    print()

In [ ]:
# ============================================================
# DESKRIPSI 3 — POIN c.2:
# Ubah satu kata petunjuk pada salah satu kalimat
# OUTPUT TABEL perbandingan ASLI vs DIUBAH
# ============================================================

print('=== EKSPERIMEN: Ubah Kata Petunjuk ===')
print()

kalimat_asli = 'Dia pergi ke [MASK] untuk meminjam buku tentang sejarah Indonesia.'
kalimat_ubah = 'Dia pergi ke [MASK] untuk membeli obat dan vitamin kesehatan.'

hasil_asli = mask_filler(kalimat_asli)
hasil_ubah = mask_filler(kalimat_ubah)

print(f'Kalimat ASLI  : {kalimat_asli}')
tabel_asli = pd.DataFrame([
    {'Peringkat': i+1, 'Prediksi': r['token_str'], 'Skor Probabilitas': round(r['score'], 4)}
    for i, r in enumerate(hasil_asli[:5])
])
display(tabel_asli)

print()
print(f'Kalimat DIUBAH: {kalimat_ubah}')
tabel_ubah = pd.DataFrame([
    {'Peringkat': i+1, 'Prediksi': r['token_str'], 'Skor Probabilitas': round(r['score'], 4)}
    for i, r in enumerate(hasil_ubah[:5])
])
display(tabel_ubah)

print()
print(f'Top-1 ASLI   \u2192 "{hasil_asli[0]["token_str"]}" (skor: {hasil_asli[0]["score"]:.4f})')
print(f'Top-1 DIUBAH \u2192 "{hasil_ubah[0]["token_str"]}" (skor: {hasil_ubah[0]["score"]:.4f})')
print()
print('\u2192 Jawaban analisis poin c.1 dan c.2: TULIS TANGAN di logbook')

In [ ]:
# ============================================================
# DESKRIPSI 4 — POIN a & b:
# Masukkan 20 kalimat ke model sentiment-analysis
# OUTPUT TABEL:
# KALIMAT | LABEL INTERPRETASI | LABEL AKTUAL | CONFIDENCE SCORE
# ============================================================

print('=' * 90)
print('EKSPERIMEN SENTIMENT ANALYSIS & DETEKSI SARKASME')
print('=' * 90)
print()

hasil_sentimen = []

for idx, row in df_sentiment.iterrows():
    kalimat      = row['kalimat']
    label_aktual = row['label']

    hasil        = sentiment_analyzer(kalimat, truncation=True, max_length=512)
    label_mentah = hasil[0]['label'].lower()
    skor         = hasil[0]['score']

    if label_mentah in ['label_0', 'neg', 'negative']:
        label_interpretasi = 'NEGATIF'
    elif label_mentah in ['label_2', 'pos', 'positive']:
        label_interpretasi = 'POSITIF'
    else:
        label_interpretasi = 'NETRAL'

    if label_aktual == 'negatif':
        label_aktual_std = 'NEGATIF'
    elif label_aktual == 'positif':
        label_aktual_std = 'POSITIF'
    else:
        label_aktual_std = 'AMBIGU'

    benar = 'BENAR \u2705' if label_interpretasi == label_aktual_std else 'SALAH \u274c'

    hasil_sentimen.append({
        'No'                 : idx + 1,
        'Kalimat'            : kalimat[:45] + '...',
        'Label Interpretasi' : label_interpretasi,
        'Label Aktual'       : label_aktual_std,
        'Confidence Score'   : round(skor, 4),
        'Hasil'              : benar
    })

# Tampilkan tabel visual
df_hasil_sentimen = pd.DataFrame(hasil_sentimen)
display(df_hasil_sentimen)
print()

# ============================================================
# DESKRIPSI 4 — POIN c.1:
# Berapa yang ditebak benar dari 20 kalimat?
# ============================================================
total_benar = sum(1 for h in hasil_sentimen if 'BENAR' in h['Hasil'])
print('=' * 50)
print(f'HASIL: {total_benar} dari 20 kalimat ditebak BENAR')
print(f'Akurasi model: {total_benar/20*100:.1f}%')
print('=' * 50)
print()
print('\u2192 Jawaban analisis poin c.2, c.3, c.4: TULIS TANGAN di logbook')

In [ ]:
# ============================================================
# DESKRIPSI 5 — POIN a & b:
# Gunakan 20 kalimat informal sebagai input model NER
# OUTPUT TABEL: KATA | KATEGORI | CONFIDENCE SCORE
# ============================================================

print('=' * 70)
print('EKSPERIMEN NAME ENTITY RECOGNITION (NER)')
print('Input: 20 Kalimat Informal')
print('=' * 70)

for idx, row in df_informal.iterrows():
    kalimat   = row['kalimat']
    hasil_ner = ner_recognizer(kalimat)

    print(f'\n[Kalimat {idx+1}]')
    print(f'Input : {kalimat}\n')

    if not hasil_ner:
        print('  \u2192 Tidak ada entitas yang terdeteksi.')
    else:
        tabel_ner = pd.DataFrame([
            {
                'Kata'            : ent['word'],
                'Kategori'        : ent['entity_group'],
                'Confidence Score': round(ent['score'], 4)
            }
            for ent in hasil_ner
        ])
        display(tabel_ner)
    print()

print('\u2192 Jawaban analisis poin c.1 dan c.2: TULIS TANGAN di logbook')